In [ ]:
!pip install -U datasets
!pip install -U sagemaker

In [ ]:
%%writefile train.py
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)
from huggingface_hub import login
import torch
import os


# Load dataset directly from Hugging Face Hub
print("Loading dataset from Hugging Face Hub...")
finetuning_dataset = load_dataset("AmiraliSH/my-qa-dataset")
train_dataset = finetuning_dataset["train"]

# Load tokenizer and model
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    text = [q + a for q, a in zip(examples["question"], examples["answer"])]
    tokenizer.truncation_side = "left"
    return tokenizer(
        text,
        truncation=True,
        padding=True,
        max_length=1024
    )

tokenized_train_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    batch_size=32,
    remove_columns=train_dataset.column_names,
)

tokenized_train_dataset = tokenized_train_dataset.add_column(
    "labels", tokenized_train_dataset["input_ids"]
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# SageMaker expects models to be saved to /opt/ml/model
output_dir = "/opt/ml/model" if "SM_MODEL_DIR" in os.environ else "./model"

training_args = TrainingArguments(
    learning_rate=1e-5,
    num_train_epochs=4,
    per_device_train_batch_size=1,
    output_dir=output_dir,
    overwrite_output_dir=True,
    save_steps=200,
    logging_strategy="epoch",
    optim="adamw_torch",
    gradient_accumulation_steps=4,
    gradient_checkpointing=False,
    save_total_limit=1,
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    data_collator=data_collator,
)

print("Starting training...")
trainer.train()

print("Saving model and tokenizer...")
# Save both model AND tokenizer
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)  # This line was missing!
print("Training completed!")

Writing train.py


In [ ]:
from sagemaker.huggingface import HuggingFace
import sagemaker

role = ""  # Replace with your actual ARN
sess = sagemaker.Session()

# Create a dummy file for input_data requirement
with open('dummy.txt', 'w') as f:
    f.write('This is a dummy file for SageMaker input requirement')

# Upload dummy file to S3
bucket = sess.default_bucket()
dummy_s3_path = sess.upload_data(path='dummy.txt', bucket=bucket, key_prefix='dummy')

# Configure the estimator
huggingface_estimator = HuggingFace(
    entry_point='train.py',
    source_dir='.',
    instance_type='ml.g5.2xlarge',
    instance_count=1,
    role=role,
    transformers_version='4.36.0',
    pytorch_version='2.1.0',
    py_version='py310',
    environment={
        'HUGGINGFACE_HUB_CACHE': '/tmp/.cache',  # Optional: cache location
    }
)

# Use dummy S3 path (your script will ignore this and use HF Hub)
input_data = {'train': dummy_s3_path}

# Start training
print("Starting SageMaker training job...")
huggingface_estimator.fit(input_data)

print("Training job completed!")
print(f"Model artifacts: {huggingface_estimator.model_data}")

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.


Starting SageMaker training job...


INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: huggingface-pytorch-training-2025-07-08-13-45-24-188


2025-07-08 13:45:29 Starting - Starting the training job
2025-07-08 13:45:29 Pending - Training job waiting for capacity...
2025-07-08 13:45:54 Pending - Preparing the instances for training...
2025-07-08 13:46:23 Downloading - Downloading input data...
2025-07-08 13:46:38 Downloading - Downloading the training image..................
2025-07-08 13:49:45 Training - Training image download completed. Training in progress..bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
/opt/conda/lib/python3.10/site-packages/paramiko/pkey.py:100: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "cipher": algorithms.TripleDES,
/opt/conda/lib/python3.10/site-packages/paramiko/transport.py:259: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be remo

In [37]:
from sagemaker.huggingface import HuggingFaceModel
import sagemaker

# Create HuggingFace Model class
huggingface_model = HuggingFaceModel(
    model_data=huggingface_estimator.model_data,  # Path to your trained model artifacts
    role=role,                                    # IAM role with permissions to create endpoint
    transformers_version='4.37.0',
    pytorch_version='2.1.0',
    py_version='py310',
    model_server_workers=1                        # Number of worker processes
)


# Deploy the model to an endpoint
predictor = huggingface_model.deploy(
    initial_instance_count=1,
    instance_type="ml.g5.2xlarge"  # Choose an appropriate instance type
)

INFO:sagemaker:Creating model with name: huggingface-pytorch-inference-2025-07-08-13-55-00-365
INFO:sagemaker:Creating endpoint-config with name huggingface-pytorch-inference-2025-07-08-13-55-01-066
INFO:sagemaker:Creating endpoint with name huggingface-pytorch-inference-2025-07-08-13-55-01-066


----------!

In [49]:
payload = {
    "inputs": "Where does Amirali currently live?",  # Use chat template
    "parameters": {
        "max_new_tokens": 50,
        "temperature": 0.7,
        "return_full_text": False  # This tells the model to omit the input
    }
}

response = predictor.predict(payload)
response

[{'generated_text': 'Turkey, Istanbul.'}]

In [43]:
response

[{'generated_text': 'Turkey, Istanbul.'}]